# Vehicle-client compatibility

In many routing problems, not every vehicle can serve every client: a client may need a vehicle with a tail lift or a crane, a refrigerated truck, or a driver with a particular certification; or a client may simply be reserved for a specific subset of the fleet.

PyVRP's `Model` lets you express these restrictions directly, in two complementary ways:

- **Skills**: give a vehicle type `skills`, and give a client `required_skills`. A client may then only be served by vehicle types that have *all* of its required skills.
- **Allowed vehicle types**: give a client an explicit list of `allowed_vehicle_types` (by object or by name).

Both are compiled into additional load dimensions when the problem is built, which makes them **hard** constraints: an incompatible assignment is infeasible, not merely discouraged.

In [ ]:
from pyvrp import Model
from pyvrp.stop import MaxRuntime

## Skills

We set up a small instance with two vehicle types. Only the second has the `crane` skill, and only one client requires it. That client must therefore be served by the crane-equipped vehicle.

In [ ]:
m = Model()
locations = [m.add_location(x=x, y=0) for x in range(4)]
depot = m.add_depot(locations[0])

m.add_client(locations[1])
m.add_client(locations[2])
m.add_client(locations[3], required_skills=["crane"])

m.add_vehicle_type(1, start_depot=depot, end_depot=depot, name="plain")
m.add_vehicle_type(
    1, start_depot=depot, end_depot=depot, skills=["crane"], name="crane"
)

for frm in locations:
    for to in locations:
        if frm is not to:
            dist = int(abs(frm.x - to.x) + abs(frm.y - to.y))
            m.add_edge(frm, to, distance=dist, duration=dist)

res = m.solve(stop=MaxRuntime(1), display=False)
data = m.data()

for route in res.best.routes():
    veh = data.vehicle_type(route.vehicle_type())
    clients = [act.idx for act in route if act.is_client()]
    print(f"{veh.name!r} vehicle serves clients {clients}")

Client 2 (the one requiring the crane) is served by the `crane` vehicle; the other clients can go on either vehicle.

## Allowed vehicle types

Alternatively, we can restrict a client to a specific set of vehicle types directly. Vehicle types can be referenced by name, so clients may even be added before their vehicle types exist.

In [ ]:
m = Model()
locations = [m.add_location(x=x, y=0) for x in range(4)]
depot = m.add_depot(locations[0])

m.add_client(locations[1])
m.add_client(locations[2])
m.add_client(locations[3], allowed_vehicle_types=["van"])

m.add_vehicle_type(1, start_depot=depot, end_depot=depot, name="truck")
m.add_vehicle_type(1, start_depot=depot, end_depot=depot, name="van")

for frm in locations:
    for to in locations:
        if frm is not to:
            dist = int(abs(frm.x - to.x) + abs(frm.y - to.y))
            m.add_edge(frm, to, distance=dist, duration=dist)

res = m.solve(stop=MaxRuntime(1), display=False)
data = m.data()

for route in res.best.routes():
    veh = data.vehicle_type(route.vehicle_type())
    clients = [act.idx for act in route if act.is_client()]
    print(f"{veh.name!r} vehicle serves clients {clients}")

Client 2 is only allowed on the `van`, so it is served there regardless of cost.

## Notes

- Each distinct skill, and each vehicle type that some client forbids, adds one load dimension. For large heterogeneous fleets, prefer the skills model, which keeps the number of extra dimensions equal to the number of skills.
- Empty `required_skills` / `allowed_vehicle_types` means *no* restriction, so existing models are unaffected.
- If a *required* client cannot be served by any vehicle type (given its skills and allowed types), building the data raises a `ValueError`.
- For a *soft* preference (avoid, but not forbid) rather than a hard restriction, use routing profiles instead; see the profiles tutorial.